# Project 1 — RDKit Molecule Analyser

This notebook builds the basic cheminformatics layer used by the later machine-learning projects. The workflow starts with a small molecule table, converts SMILES strings into RDKit molecule objects, checks valid and invalid structures, calculates simple descriptors and searches for functional groups with SMARTS.

The notebook is deliberately small and explicit. Before using large public datasets, it is safer to understand what RDKit is doing to each molecule: how a string becomes a `Mol`, how atoms and bonds are stored, why invalid structures must be removed and how descriptors become a clean table.

By the end, the analysed table can be reused as a small reference dataset for quick molecule inspection, filtering and visualisation.

## 1. Setup

RDKit is the core chemistry toolkit in this project. It turns molecular strings into chemistry-aware objects, while pandas keeps the results in tables. Matplotlib is used only for simple visual checks.

A useful habit is to keep imports grouped by role: table handling, plotting and chemistry functions. That makes it easier to see which package is responsible for each step later in the notebook.

In [ ]:
# Optional Colab install: !pip install rdkit-pypi

import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, rdMolDescriptors

### Exercise — create a small environment report

Write your own short environment report. Include at least the package names being used, one sentence describing the role of each package and a small check that RDKit can parse ethanol.

Guidance: keep the result as a pandas table so it is easy to read in the notebook. This is a useful first diagnostic cell when sharing notebooks publicly.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create a small table with package names and their role in this notebook.
    # Add at least: pandas, matplotlib, RDKit.


    # Parse ethanol and store whether parsing was successful.


    # Display the environment report.


    pass
else:
    print("Set RUN_EXERCISE = True after writing your environment-report code.")


### Answer — create a small environment report

Reference solution.

In [ ]:
# Check that RDKit can parse a simple molecule.
ethanol_mol = Chem.MolFromSmiles("CCO")

# Store package roles in a readable table.
environment_report = pd.DataFrame([
    {"package": "pandas", "role": "store molecule tables"},
    {"package": "matplotlib", "role": "draw simple descriptor plots"},
    {"package": "RDKit", "role": "parse and analyse molecular structures"},
])

environment_report["ethanol_parse_ok"] = ethanol_mol is not None

environment_report


## 2. Build a small molecule table

A molecule table normally needs at least two columns: a readable compound name and a SMILES string. A small hand-built table is useful at the start because every row can be inspected manually.

This part uses common small organic molecules, drugs and fragrance-like compounds. The variety is enough to show alcohols, carbonyls, aromatics, acids, esters and hydrophobic molecules without making the first notebook too heavy.

In [ ]:
molecule_data = [
    {"name": "ethanol", "smiles": "CCO"},
    {"name": "acetone", "smiles": "CC(=O)C"},
    {"name": "benzene", "smiles": "c1ccccc1"},
    {"name": "aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O"},
    {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C"},
    {"name": "invalid_example", "smiles": "ABCXYZ"},
]

# Store molecule names and SMILES strings in a table.
df = pd.DataFrame(molecule_data)

# Inspect the first rows before doing any chemistry operations.
print("Input shape:", df.shape)
df.head()

### Exercise — build your own mini molecule panel

Create a new molecule table with at least six compounds. Include at least one alcohol, one aromatic compound, one acid or ester and one deliberately invalid SMILES string.

Guidance: this tests whether the later parsing code can separate valid structures from invalid input. Use your own compound choices rather than only copying the example table.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create a list of dictionaries with name and SMILES keys.
    # Include at least six rows and one deliberately invalid SMILES.


    # Convert the list into a dataframe.


    # Display the new table.


    pass
else:
    print("Set RUN_EXERCISE = True after creating your own molecule panel.")


### Answer — build your own mini molecule panel

Reference solution.

In [ ]:
# Build a small molecule panel with one invalid row for validation practice.
custom_molecules = [
    {"name": "methanol", "smiles": "CO"},
    {"name": "ethyl acetate", "smiles": "CCOC(=O)C"},
    {"name": "benzoic acid", "smiles": "O=C(O)c1ccccc1"},
    {"name": "vanillin", "smiles": "COc1cc(C=O)ccc1O"},
    {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C"},
    {"name": "invalid_example", "smiles": "not_a_smiles"},
]

custom_df = pd.DataFrame(custom_molecules)
custom_df


## 3. Parse SMILES into RDKit molecules

`Chem.MolFromSmiles()` is the key boundary between text data and molecular data. A SMILES string is just text; an RDKit `Mol` object contains atoms, bonds, aromaticity information and valence checks.

Invalid SMILES strings return `None`, so parsing should always be followed by a validity check. Downstream descriptor calculation should only use rows with valid molecule objects.

In [ ]:
# Convert each SMILES string into an RDKit Mol object.
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)

# Mark rows that were successfully parsed.
df["is_valid"] = df["mol"].notnull()

# Keep a clean table for downstream RDKit operations.
valid_df = df[df["is_valid"]].copy()

print(df[["name", "smiles", "is_valid"]])
print("Valid molecules:", len(valid_df))

In [ ]:
# Canonical SMILES gives one standard RDKit representation per molecule.
valid_df["canonical_smiles"] = valid_df["mol"].apply(lambda mol: Chem.MolToSmiles(mol, canonical=True))

# Molecular formula is a quick chemical sanity check.
valid_df["formula"] = valid_df["mol"].apply(rdMolDescriptors.CalcMolFormula)

valid_df[["name", "canonical_smiles", "formula"]]

### Exercise — write a reusable SMILES audit function

Write a function that receives a dataframe with a SMILES column, parses each molecule and returns a short audit table. The audit should report the number of rows, valid molecules, invalid molecules and valid percentage.

Guidance: reusable audit functions make later notebooks safer because the same check can be applied to public datasets before descriptor calculation.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    def audit_smiles_table(input_df, smiles_column="smiles"):
        # Parse molecules from the selected SMILES column.


        # Count total, valid and invalid rows.


        # Return a one-row dataframe with the audit results.


        pass

    # Run your function on df or on a custom molecule table.

else:
    print("Set RUN_EXERCISE = True after writing audit_smiles_table().")


### Answer — write a reusable SMILES audit function

Reference solution.

In [ ]:
def audit_smiles_table(input_df, smiles_column="smiles"):
    # Parse the SMILES column into RDKit Mol objects.
    parsed_mols = input_df[smiles_column].apply(lambda s: Chem.MolFromSmiles(str(s)))

    total_rows = len(input_df)
    valid_rows = parsed_mols.notnull().sum()
    invalid_rows = total_rows - valid_rows
    valid_percent = 100 * valid_rows / total_rows if total_rows else 0

    return pd.DataFrame([{
        "total_rows": total_rows,
        "valid_rows": valid_rows,
        "invalid_rows": invalid_rows,
        "valid_percent": round(valid_percent, 1),
    }])

audit_smiles_table(df)


## 4. Inspect atoms and bonds

A molecule object stores atom-level and bond-level information. Atom inspection is useful for checking element composition, aromatic atoms and formal charges. Bond inspection is useful for checking bond order, aromatic bonds and connectivity.

This is the structural layer underneath descriptors and fingerprints. Later ML models do not see chemistry directly unless it is converted into features from these atom and bond patterns.

In [ ]:
aspirin_mol = valid_df.loc[valid_df["name"] == "aspirin", "mol"].iloc[0]

atom_rows = []
for atom in aspirin_mol.GetAtoms():
    atom_rows.append({
        "atom_index": atom.GetIdx(),
        "symbol": atom.GetSymbol(),
        "atomic_number": atom.GetAtomicNum(),
        "degree": atom.GetDegree(),
        "is_aromatic": atom.GetIsAromatic(),
    })

# Convert atom objects into a readable table.
atom_df = pd.DataFrame(atom_rows)
atom_df.head(10)

In [ ]:
bond_rows = []
for bond in aspirin_mol.GetBonds():
    bond_rows.append({
        "begin_atom": bond.GetBeginAtomIdx(),
        "end_atom": bond.GetEndAtomIdx(),
        "bond_type": str(bond.GetBondType()),
        "is_aromatic": bond.GetIsAromatic(),
    })

# Bond tables help confirm connectivity and aromaticity.
bond_df = pd.DataFrame(bond_rows)
bond_df.head(10)

### Exercise — summarise atoms and bonds for any molecule

Write a function that accepts one RDKit molecule and returns two tables: an atom table and a bond table. Include atom index, element symbol, aromatic flag and degree in the atom table. Include begin atom, end atom and bond type in the bond table.

Guidance: this exercise makes the molecule object less abstract. Most descriptors, fingerprints and graph models are ultimately derived from atom and bond information.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    def summarise_molecule_graph(mol):
        # Build a list of atom dictionaries.


        # Build a list of bond dictionaries.


        # Return two dataframes.


        pass

    # Choose one molecule from valid_df and run your function.

else:
    print("Set RUN_EXERCISE = True after writing summarise_molecule_graph().")


### Answer — summarise atoms and bonds for any molecule

Reference solution.

In [ ]:
def summarise_molecule_graph(mol):
    atom_rows = []
    for atom in mol.GetAtoms():
        atom_rows.append({
            "atom_index": atom.GetIdx(),
            "symbol": atom.GetSymbol(),
            "is_aromatic": atom.GetIsAromatic(),
            "degree": atom.GetDegree(),
        })

    bond_rows = []
    for bond in mol.GetBonds():
        bond_rows.append({
            "begin_atom": bond.GetBeginAtomIdx(),
            "end_atom": bond.GetEndAtomIdx(),
            "bond_type": str(bond.GetBondType()),
            "is_aromatic": bond.GetIsAromatic(),
        })

    return pd.DataFrame(atom_rows), pd.DataFrame(bond_rows)

example_mol = valid_df.loc[valid_df["name"] == "aspirin", "mol"].iloc[0]
example_atoms, example_bonds = summarise_molecule_graph(example_mol)

display(example_atoms.head())
display(example_bonds.head())


## 5. Draw molecules

Visualising structures is a simple quality-control step. A drawn molecule can quickly reveal whether a SMILES string was parsed as expected, whether aromatic rings are present and whether a selected compound is the intended one.

Drawing is especially useful before interpreting descriptors or SMARTS matches, because numerical values are easier to understand when the structure is visible.

In [ ]:
# Draw a single molecule with a legend.
Draw.MolToImage(aspirin_mol, size=(350, 250), legend="aspirin")

In [ ]:
# Select the first valid molecules for a grid view.
grid_mols = valid_df["mol"].head(5).tolist()
grid_names = valid_df["name"].head(5).tolist()

# A grid makes it easier to compare several structures at once.
Draw.MolsToGridImage(grid_mols, legends=grid_names, molsPerRow=3, subImgSize=(220, 180))

### Exercise — draw a molecule subset chosen by a rule

Create your own rule for selecting molecules to draw. For example, draw all aromatic molecules, all molecules with `LogP > 2` after descriptors are calculated or a named list of compounds.

Guidance: selecting by rule is more useful than manually drawing the same fixed examples every time. It connects visual inspection to dataframe filtering.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Choose a rule to select molecules from valid_df.


    # Store selected molecule objects and labels.


    # Draw the selected molecules in a grid.


    pass
else:
    print("Set RUN_EXERCISE = True after writing a molecule-selection rule.")


### Answer — draw a molecule subset chosen by a rule

Reference solution.

In [ ]:
# Select aromatic molecules using a simple RDKit check.
aromatic_mask = valid_df["mol"].apply(lambda mol: any(atom.GetIsAromatic() for atom in mol.GetAtoms()))
aromatic_df = valid_df[aromatic_mask].copy()

# Draw up to eight selected molecules with their names.
Draw.MolsToGridImage(
    aromatic_df["mol"].head(8).tolist(),
    legends=aromatic_df["name"].head(8).tolist(),
    molsPerRow=4,
    subImgSize=(220, 180),
)


## 6. Calculate molecular descriptors

Molecular descriptors convert structures into numerical values. Simple descriptors such as molecular weight, LogP, TPSA, hydrogen-bond donors and acceptors are interpretable and often useful for first-pass molecular analysis.

These descriptors are not a complete chemical description, but they are easy to inspect and are suitable for beginner QSAR workflows.

In [ ]:
def calculate_basic_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "RingCount": Descriptors.RingCount(mol),
    }

# Calculate descriptors row by row.
descriptor_df = valid_df["mol"].apply(calculate_basic_descriptors).apply(pd.Series)

# Join descriptor columns back to the molecule table.
analysed_df = pd.concat([valid_df.reset_index(drop=True), descriptor_df], axis=1)

analysed_df[["name", "MolWt", "LogP", "TPSA", "HBD", "HBA", "RingCount"]]

### Exercise — build a descriptor function and rank molecules

Write a function that calculates at least five descriptors for a molecule, then apply it to the valid molecule table. Use the result to rank molecules by two different descriptor columns.

Guidance: descriptor functions are the bridge between chemistry and machine-learning tables. A good descriptor table should be numeric, inspectable and easy to sort.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    def calculate_custom_descriptors(mol):
        # Return a dictionary of at least five descriptor values.


        pass

    # Apply the function to each valid molecule.


    # Combine descriptor values with molecule names.


    # Sort by two descriptor columns of your choice.


    pass
else:
    print("Set RUN_EXERCISE = True after writing calculate_custom_descriptors().")


### Answer — build a descriptor function and rank molecules

Reference solution.

In [ ]:
def calculate_custom_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotBonds": Lipinski.NumRotatableBonds(mol),
    }

custom_desc_df = pd.DataFrame(valid_df["mol"].apply(calculate_custom_descriptors).tolist())
custom_desc_df = pd.concat([valid_df[["name", "smiles"]].reset_index(drop=True), custom_desc_df], axis=1)

display(custom_desc_df.sort_values("LogP", ascending=False).head(5))
display(custom_desc_df.sort_values("TPSA", ascending=False).head(5))


## 7. Apply simple drug-likeness checks

Rule-based filters summarise rough molecular-property boundaries. They are not strict pass/fail truth, but they help identify molecules that are unusually large, very lipophilic or highly polar.

The aim here is to practise turning descriptor values into interpretable flags. Later ML projects use the same idea when converting molecular features into model inputs.

In [ ]:
def lipinski_summary(row):
    checks = {
        "MolWt_le_500": row["MolWt"] <= 500,
        "LogP_le_5": row["LogP"] <= 5,
        "HBD_le_5": row["HBD"] <= 5,
        "HBA_le_10": row["HBA"] <= 10,
    }
    return pd.Series({
        "lipinski_passes": sum(checks.values()),
        "lipinski_ok": all(checks.values()),
    })

# Apply the checks to every molecule.
lipinski_df = analysed_df.apply(lipinski_summary, axis=1)

# Combine descriptor values and Lipinski results.
analysed_df = pd.concat([analysed_df, lipinski_df], axis=1)

analysed_df[["name", "MolWt", "LogP", "HBD", "HBA", "lipinski_passes", "lipinski_ok"]]

## 8. Detect functional groups with SMARTS

SMARTS patterns describe substructures. They allow the notebook to ask chemistry questions such as whether a molecule contains an alcohol, acid, ester, amine or aromatic ring.

Functional-group detection is useful because many chemical behaviours are linked to recognisable substructures. It also introduces substructure search before fingerprints and graph models.

In [ ]:
smarts_patterns = {
    "alcohol": "[OX2H][CX4]",
    "carboxylic_acid": "C(=O)[OX2H1]",
    "ester": "C(=O)O[#6]",
    "amine": "[NX3;H2,H1;!$(NC=O)]",
    "aromatic_ring": "a1aaaaa1",
}

# Convert SMARTS strings into RDKit query molecules.
smarts_queries = {name: Chem.MolFromSmarts(pattern) for name, pattern in smarts_patterns.items()}

def detect_functional_groups(mol):
    return {name: mol.HasSubstructMatch(query) for name, query in smarts_queries.items()}

# Add functional-group flags to the descriptor table.
fg_df = analysed_df["mol"].apply(detect_functional_groups).apply(pd.Series)
analysed_df = pd.concat([analysed_df, fg_df], axis=1)

analysed_df[["name", "alcohol", "carboxylic_acid", "ester", "amine", "aromatic_ring"]]

### Exercise — design a functional-group screen

Create a SMARTS dictionary with at least six functional-group patterns. Apply it to the valid molecule table and count how many molecules match each pattern.

Guidance: the goal is not to make a perfect expert substructure library. The aim is to practise converting chemistry questions into SMARTS queries and summarising matches in a table.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create a dictionary mapping group names to SMARTS strings.


    # Convert each SMARTS string into an RDKit query molecule.


    # Apply each query to every molecule and store True/False results.


    # Count matches for each functional group.


    pass
else:
    print("Set RUN_EXERCISE = True after writing your SMARTS screen.")


### Answer — design a functional-group screen

Reference solution.

In [ ]:
custom_smarts = {
    "alcohol": "[OX2H]",
    "carboxylic_acid": "C(=O)[OX2H1]",
    "ester": "C(=O)O[#6]",
    "aldehyde": "[CX3H1](=O)[#6]",
    "amine": "[NX3;H2,H1,H0]",
    "aromatic_ring": "a1aaaaa1",
}

fg_results = valid_df[["name", "smiles", "mol"]].copy()
for group_name, smarts in custom_smarts.items():
    query = Chem.MolFromSmarts(smarts)
    fg_results[group_name] = fg_results["mol"].apply(lambda mol: mol.HasSubstructMatch(query))

fg_counts = fg_results[list(custom_smarts)].sum().sort_values(ascending=False)

display(fg_results.drop(columns="mol").head())
display(fg_counts.to_frame("match_count"))


## 9. Visualise descriptor behaviour

Plots help check relationships between molecular properties. For example, molecular weight and LogP often move together for hydrocarbon-rich molecules, while TPSA captures polarity and hydrogen-bonding potential.

These plots are not formal models. They are quick checks for trends, outliers and molecules worth inspecting more closely.

In [ ]:
# Plot molecular size against lipophilicity.
plt.figure(figsize=(7, 5))
plt.scatter(analysed_df["MolWt"], analysed_df["LogP"])

# Label each point with the molecule name.
for _, row in analysed_df.iterrows():
    plt.text(row["MolWt"], row["LogP"], row["name"], fontsize=8)

plt.xlabel("Molecular weight")
plt.ylabel("LogP")
plt.title("Molecular weight vs LogP")
plt.show()

### Exercise — make a two-panel descriptor check

Create two plots that inspect different descriptor relationships. Choose one plot involving LogP and another involving TPSA, HBD or HBA. Label axes clearly and add molecule names only if the plot remains readable.

Guidance: the best plot is not always the most complex one. A useful descriptor plot should make one chemical relationship easier to inspect.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create the first descriptor plot.



    # Create the second descriptor plot.



    pass
else:
    print("Set RUN_EXERCISE = True after writing two descriptor plots.")


### Answer — make a two-panel descriptor check

Reference solution.

In [ ]:
# Plot molecular size against lipophilicity.
plt.figure(figsize=(7, 5))
plt.scatter(valid_df["MolWt"], valid_df["LogP"])
plt.xlabel("Molecular weight")
plt.ylabel("LogP")
plt.title("Molecular weight vs LogP")
plt.show()

# Plot molecular polarity against lipophilicity.
plt.figure(figsize=(7, 5))
plt.scatter(valid_df["TPSA"], valid_df["LogP"])
plt.xlabel("TPSA")
plt.ylabel("LogP")
plt.title("TPSA vs LogP")
plt.show()


## 10. Export the analysed table

Exporting the cleaned table makes the analysis reusable. A CSV file can be opened in spreadsheet software, committed to GitHub as a small example output or loaded into later notebooks.

Before exporting, keep only columns that are useful outside the notebook. RDKit `Mol` objects themselves are Python objects and are usually not written directly into CSV files.

In [ ]:
output_columns = [
    "name", "smiles", "canonical_smiles", "formula",
    "MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds",
    "lipinski_passes", "lipinski_ok",
    "alcohol", "carboxylic_acid", "ester", "amine", "aromatic_ring",
]

# Save the clean analysis table to the current working directory.
analysed_df[output_columns].to_csv("project1_rdkit_molecule_analysis.csv", index=False)

# Read the file back to confirm the export worked.
reloaded_df = pd.read_csv("project1_rdkit_molecule_analysis.csv")
reloaded_df.head()

### Exercise — export a filtered analysis table

Create a clean export table containing molecule names, canonical SMILES, descriptors and at least one functional-group flag. Filter it to a subset you think is useful, then save it as a CSV.

Guidance: a good export table should avoid Python-only objects such as RDKit `Mol` columns. Keep values that can be reused outside the notebook.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Select export columns that do not include RDKit Mol objects.


    # Apply your own filter.


    # Save the filtered table to CSV.


    pass
else:
    print("Set RUN_EXERCISE = True after preparing your export table.")


### Answer — export a filtered analysis table

Reference solution.

In [ ]:
# Select columns that are safe to write into a CSV file.
export_columns = [
    "name", "canonical_smiles", "MolWt", "LogP", "TPSA", "HBD", "HBA",
]

export_df = valid_df[export_columns].copy()

# Example filter: keep molecules with moderate molecular weight.
filtered_export_df = export_df.query("100 <= MolWt <= 350").copy()

filtered_export_df.to_csv("project1_filtered_molecule_analysis.csv", index=False)
filtered_export_df.head()
